## Homework 9: Text Classification with Fine-Tuned BERT

### Due: Midnight on November 1st (with 2-hour grace period) — Worth 85 points

In this final homework, we’ll explore **fine-tuning pretrained Transformer models** for text classification using the **IMDB Movie Review** dataset. You’ll begin by training and evaluating a DistilBERT baseline, then conduct experiments to investigate how training configurations, dataset size, and model architecture affect predictive performance and computational cost.

You’ll complete three problems:

* **Problem 1:** Compare three context-length training configurations and their validation performance and training times. You’ll also compare your best configuration with the initial DistilBERT baseline.
* **Problem 2:** Measure how **training data size** affects both model performance and total training time.
* **Problem 3:** Compare **DistilBERT and BERT-base** to analyze the trade-offs between model size, accuracy, and training time.

In each problem, you’ll report experimental results, interpret the differences, and justify a choice based on validation performance and computational cost.

> **Note:** This homework was developed and tested on **Google Colab**, due to version conflicts when running locally. It is **strongly recommended** that you complete your work on Colab as well.

There are three problems with six graded questions in total. Each graded question is worth 14 points, with one additional point for completing the homework.


### Developing with Increasing Amounts of Data

Fine-tuning Transformer models can be computationally expensive. In this assignment, you will use a **progressive development workflow**: first verify that your code works with a small dataset, then increase the amount of training data as your experiments become more reliable.

The global `DATA_MODE` setting controls the training subset used for initial model development:

- `"smoke"` — 1% of training data, for checking that your code runs.
- `"dev"` — 10% of training data, for debugging and development.
- `"baseline"` — 25% of training data, for establishing a meaningful baseline.
- `"full"` — 100% of the training pool, when full-data training is needed.

The subsets are reproducible and nested: larger subsets contain the examples used in smaller subsets.

**Important:** The development setting affects only the training data. The validation and test sets remain fixed, so evaluation results are comparable. Do not use the test set to select models or tune hyperparameters.

The graded experiments in Problems 1–3 specify their own training-data fractions, independently of `DATA_MODE`. Changing this global setting does not change the datasets used in those experiments.

For the baseline, use `DATA_MODE = "baseline"`. For the graded experiments, follow the settings specified in each problem.


In [1]:
# Install once per new Colab runtime
%pip -q install -U keras keras-hub tensorflow tensorflow-text datasets evaluate

# ---- Backend BEFORE importing keras ----
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import time
import random
import numpy as np
import keras
import keras_hub as kh
import evaluate
from datasets import load_dataset, Dataset, Features, Value, ClassLabel
from sklearn.model_selection import train_test_split

from keras import mixed_precision                    # generally faster
mixed_precision.set_global_policy("mixed_float16")

Note: you may need to restart the kernel to use updated packages.


I0000 00:00:1793228687.203397    3343 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793228687.511876    3343 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/caleb/miniconda3/envs/tf_gpu/lib/python3.12/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
I0000 00:00:1793228688.775314    3343 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different compu

### Here is where you can set global hyperparameters for this homework

In [27]:

# ------------ Global Configuration ------------

SEED = 42

# Model and training settings
MAX_LEN = 128
EPOCHS = 2
BATCH = 32
EVAL_BATCH = 64


# Development workflow
# Use "smoke" for debugging, "dev" for development,
# "baseline" for 25%, or "full" for 100%.

DATA_MODE = "baselineplus"

DATA_FRACTIONS = {
    "smoke":    0.01,   # 1% of training pool
    "dev":      0.10,   # 10% of training pool
    "baseline": 0.25,   # 25% of training pool
    "baselineplus": 0.50,
    "full":     1.00    # Entire training pool
}

assert DATA_MODE in DATA_FRACTIONS
TRAIN_FRAC = DATA_FRACTIONS[DATA_MODE]


keras.utils.set_random_seed(SEED)

print(f"Data mode: {DATA_MODE}")
print(f"Training fraction: {TRAIN_FRAC:.0%}")


Data mode: baselineplus
Training fraction: 50%


### Load and Preprocess the IMDB Movie Review Dataset

In [28]:

# ------------ Load and Partition IMDB ------------

imdb = load_dataset("stanfordnlp/imdb")

# Preserve the original held-out test set
test_ds = imdb["test"]

# Create a fixed, stratified validation set
indices = np.arange(len(imdb["train"]))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.10,
    random_state=SEED,
    stratify=np.array(imdb["train"]["label"])
)

train_pool = imdb["train"].select(train_idx.tolist())
val_ds = imdb["train"].select(val_idx.tolist())

# ------------ Reproducible Nested Training Subsets ------------

# Shuffle each class independently, then interleave classes.
# This gives balanced, nested training subsets.
rng = np.random.default_rng(SEED)

labels = np.array(train_pool["label"])
neg_idx = np.flatnonzero(labels == 0)
pos_idx = np.flatnonzero(labels == 1)

rng.shuffle(neg_idx)
rng.shuffle(pos_idx)

train_order = np.column_stack((neg_idx, pos_idx)).ravel()

def get_training_subset(frac):
    """Return a balanced, nested fraction of the training pool."""
    if not 0 < frac <= 1:
        raise ValueError("frac must be in (0, 1].")

    n_pairs = max(1, round(len(train_order) * frac / 2))
    selected = train_order[:2 * n_pairs]

    return train_pool.select(selected.tolist())

# Apply the development throttle
train_ds = get_training_subset(TRAIN_FRAC)

# ------------ NumPy Arrays for Keras ------------

X_tr = np.array(train_ds["text"], dtype=object)
y_tr = np.array(train_ds["label"], dtype="int32")

X_va = np.array(val_ds["text"], dtype=object)
y_va = np.array(val_ds["label"], dtype="int32")

X_te = np.array(test_ds["text"], dtype=object)
y_te = np.array(test_ds["label"], dtype="int32")

# ------------ Summary ------------

def _counts(ds):
    values = np.array(ds["label"], dtype=int)
    return len(values), np.bincount(values, minlength=2).tolist()

print(f"Data mode: {DATA_MODE} ({TRAIN_FRAC:.0%})")
print("Training pool:", _counts(train_pool))
print("Training subset:", _counts(train_ds))
print("Validation:", _counts(val_ds))
print("Test:", _counts(test_ds))


Data mode: baselineplus (50%)
Training pool: (22500, [11250, 11250])
Training subset: (11250, [5625, 5625])
Validation: (2500, [1250, 1250])
Test: (25000, [12500, 12500])


### Build and train a baseline Distil-Bert Text Classifier

In [4]:

# ---- DistilBERT baseline: develop using the selected data throttle ----

print(f"Data mode: {DATA_MODE}")
print(f"Training examples: {len(X_tr):,}")
print(f"Validation examples: {len(X_va):,}")
print(f"Maximum sequence length: {MAX_LEN}")
print(f"Batch size: {BATCH}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)

preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length=MAX_LEN
)

model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes=2,
    preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))


Data mode: baseline
Training examples: 5,624
Validation examples: 2,500
Maximum sequence length: 128
Batch size: 32


100%|██████████| 462/462 [00:00<00:00, 1.38MB/s]


100%|██████████| 794/794 [00:00<00:00, 3.03MB/s]


100%|██████████| 226k/226k [00:00<00:00, 1.53MB/s]
I0000 00:00:1793228712.973621    3343 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5527 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.9


100%|██████████| 253M/253M [00:06<00:00, 38.4MB/s] 


...

176/176 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - acc: 0.7628 - loss: 0.4949

I0000 00:00:1793228779.907789    3684 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_31170__.70
I0000 00:00:1793228780.504264    4894 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793228780.788358    4896 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_98', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793228781.115927    4895 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_100', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793228782.986682    3684 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_476', 58 bytes spill stores, 58 bytes spill loads

I0000 00:00:1793228784.514928    3683 dot_merger.cc:481] Mergi

176/176 ━━━━━━━━━━━━━━━━━━━━ 66s 222ms/step - acc: 0.7628 - loss: 0.4949 - val_acc: 0.8456 - val_loss: 0.3561
Epoch 2/2
176/176 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - acc: 0.8606 - loss: 0.3284 - val_acc: 0.8464 - val_loss: 0.3624


I0000 00:00:1793228802.108844    3682 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_31170__.70
I0000 00:00:1793228803.491213    5383 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_100', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793228803.883683    5391 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_98', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793228804.247381    5383 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793228806.021185    3682 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_452', 58 bytes spill stores, 58 bytes spill loads

I0000 00:00:1793228808.623284    3682 dot_merger.cc:481] Mergi


Best epoch: 1
Validation loss (restored model): 0.3561
Validation accuracy: 0.8456
Validation F1: 0.8457

Validation confusion matrix:
[[1056  194]
 [ 192 1058]]

Training time: 00:01:19



# Problem 1 — Mini Sweep: Context Length and Training Cost (3 Runs)

In this problem, we'll investigate how **context length** (`MAX_LEN`) affects the performance and computational cost of fine-tuning DistilBERT.

A longer context allows the model to use more of each text, potentially improving its predictions. However, longer sequences require more computation and memory, particularly because of the attention mechanism.

We'll compare three configurations, using different batch sizes and learning rates appropriate for each context length.

## Setup (keep these fixed)

* `SUBSET_FRAC = 0.25` — use 25% of the training dataset.
* `EPOCHS = 2` — maximum number of training epochs.
* **EarlyStopping** with `restore_best_weights=True`.
* Same random seed for all runs.
* Same training and validation examples for all runs (don't reshuffle or resplit between runs).

## Run these 3 configurations

| MAX_LEN | Learning rate | Batch size |
|---|---|---|
| 128 | `2e-5` | 32 |
| 256 | `1e-5` | 16 |
| 512 | `7.5e-6` | 8 |

These configurations represent progressively longer contexts, with smaller batch sizes to accommodate the increased memory requirements.

**Important:** Because learning rate and batch size also change, this experiment compares three practical training configurations. It does not isolate the effect of context length alone.

For each run, record:

* Best validation loss and the epoch at which it occurred.
* Validation accuracy at that epoch.
* Total training time.

Identify the configuration with the **lowest validation loss**. Problems 2 and 3 use their own fixed configurations to isolate the effects being studied.

## If you encounter an Out of Memory error

* At `MAX_LEN = 256`, reduce `BATCH` from 16 to 8.
* At `MAX_LEN = 512`, reduce `BATCH` from 8 to 4.

If you make such a change, record it when discussing your results.

Then answer the graded questions.


### Max Len 128, LR 2e-5, Batch Size 32

In [ ]:
p1a_max_len = 128
p1a_learning_rate = 2e-5
p1a_batch_size = 32

print(f"Data mode: {DATA_MODE}")
print(f"Training examples: {len(X_tr):,}")
print(f"Validation examples: {len(X_va):,}")
print(f"Maximum sequence length: {p1a_max_len}")
print(f"Batch size: {p1a_batch_size}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)

preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length=p1a_max_len
)

model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes=2,
    preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(p1a_learning_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=p1a_batch_size,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))


Data mode: baseline
Training examples: 5,624
Validation examples: 2,500
Maximum sequence length: 128
Batch size: 32
Epoch 1/2


/home/caleb/miniconda3/envs/tf_gpu/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1793229382.454416    3681 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_67124__.361
I0000 00:00:1793229391.942541    3681 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_3094', 58 bytes spill stores, 58 bytes spill loads



175/176 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - acc: 0.7800 - loss: 0.4617

I0000 00:00:1793229405.732788    3683 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_67124__.361
I0000 00:00:1793229415.696560    3683 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_3124', 58 bytes spill stores, 58 bytes spill loads



176/176 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - acc: 0.7802 - loss: 0.4610

I0000 00:00:1793229416.988571    3682 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_69838__.70
I0000 00:00:1793229418.390727    3682 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_476', 58 bytes spill stores, 58 bytes spill loads

I0000 00:00:1793229420.059144    3683 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_69838__.70
I0000 00:00:1793229421.530533    3683 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_500', 58 bytes spill stores, 58 bytes spill loads



176/176 ━━━━━━━━━━━━━━━━━━━━ 50s 170ms/step - acc: 0.7802 - loss: 0.4610 - val_acc: 0.8512 - val_loss: 0.3373
Epoch 2/2
176/176 ━━━━━━━━━━━━━━━━━━━━ 13s 70ms/step - acc: 0.8730 - loss: 0.3028 - val_acc: 0.8452 - val_loss: 0.3566


I0000 00:00:1793229435.357044    3682 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_69838__.70
I0000 00:00:1793229436.655598    3682 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_452', 58 bytes spill stores, 58 bytes spill loads

I0000 00:00:1793229439.308291    3684 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_75363__.39
I0000 00:00:1793229440.627554    3684 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_446', 58 bytes spill stores, 58 bytes spill loads

I0000 00:00:1793229442.608841    3682 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_76911__.63
I0000 00:00:1793229443.745406    3682 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_494', 58 bytes spill stores, 58 bytes spill loads




Best epoch: 1
Validation loss (restored model): 0.3373
Validation accuracy: 0.8516
Validation F1: 0.8502

Validation confusion matrix:
[[1076  174]
 [ 197 1053]]

Training time: 00:01:03


### Max Len 256, LR 1e-5, Batch Size 16

In [6]:
p1b_max_len = 256
p1b_learning_rate = 1e-5
p1b_batch_size = 16

print(f"Data mode: {DATA_MODE}")
print(f"Training examples: {len(X_tr):,}")
print(f"Validation examples: {len(X_va):,}")
print(f"Maximum sequence length: {p1b_max_len}")
print(f"Batch size: {p1b_batch_size}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)

preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length=p1b_max_len
)

model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes=2,
    preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(p1b_learning_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=p1b_batch_size,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))


Data mode: baseline
Training examples: 5,624
Validation examples: 2,500
Maximum sequence length: 256
Batch size: 16
Epoch 1/2


/home/caleb/miniconda3/envs/tf_gpu/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1793229494.429350    3685 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_105792__.361
I0000 00:00:1793229495.412100   10026 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_24', 72 bytes spill stores, 72 bytes spill loads

I0000 00:00:1793229495.932613   10024 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_26', 12 bytes spill stores, 12 bytes spill loads

I0000 00:00:1793229495.955443   10032 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local m

351/352 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - acc: 0.8321 - loss: 0.3844

I0000 00:00:1793229538.377756    3685 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_105792__.361
I0000 00:00:1793229539.773312   10594 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_115', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1793229540.361717   10603 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_27', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1793229541.352674   10603 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_280', 64 bytes spill stores, 64 bytes spill loads

I0000 00:00:1793229541.629618   10596 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_24', 60 bytes spill stores, 60 bytes spill loads

I0000 00:00:1793229541.997907   10589 subprocess_compilation.

352/352 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - acc: 0.8320 - loss: 0.3846

I0000 00:00:1793229555.955288    3685 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_109210__.70
I0000 00:00:1793229556.511928   11129 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793229556.874044   11138 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_98', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793229557.411441   11143 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_100', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793229559.346423    3685 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_440', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793229562.682881    3684 dot_merger.cc:481] Merg

352/352 ━━━━━━━━━━━━━━━━━━━━ 83s 157ms/step - acc: 0.8320 - loss: 0.3846 - val_acc: 0.8720 - val_loss: 0.3108
Epoch 2/2
352/352 ━━━━━━━━━━━━━━━━━━━━ 28s 79ms/step - acc: 0.9145 - loss: 0.2262 - val_acc: 0.8880 - val_loss: 0.2939


I0000 00:00:1793229595.794131   11652 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793229595.950346   11651 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_31', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1793229596.276718   11651 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_98', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793229596.819004   11665 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_100', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793229597.240424   11659 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3', 436 bytes spill stores, 43


Best epoch: 2
Validation loss (restored model): 0.2939
Validation accuracy: 0.8880
Validation F1: 0.8879

Validation confusion matrix:
[[1111  139]
 [ 141 1109]]

Training time: 00:01:51


### Max Len 512, LR 7.5e-6, Batch Size 8

In [7]:
p1c_max_len = 512
p1c_learning_rate = 7.5e-6
p1c_batch_size = 8

print(f"Data mode: {DATA_MODE}")
print(f"Training examples: {len(X_tr):,}")
print(f"Validation examples: {len(X_va):,}")
print(f"Maximum sequence length: {p1c_max_len}")
print(f"Batch size: {p1c_batch_size}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)

preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length=p1b_max_len
)

model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes=2,
    preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(p1c_learning_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=p1c_batch_size,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))


Data mode: baseline
Training examples: 5,624
Validation examples: 2,500
Maximum sequence length: 512
Batch size: 8
Epoch 1/2


/home/caleb/miniconda3/envs/tf_gpu/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1793229623.890822    3681 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_146700__.361
I0000 00:00:1793229633.362927    3681 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_2993', 32 bytes spill stores, 32 bytes spill loads



703/703 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - acc: 0.8327 - loss: 0.3824

I0000 00:00:1793229661.579347    3680 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_151522__.70
I0000 00:00:1793229662.169688   12269 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793229662.460869   12284 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_98', 440 bytes spill stores, 440 bytes spill loads

I0000 00:00:1793229662.840368   12280 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_100', 436 bytes spill stores, 436 bytes spill loads

I0000 00:00:1793229664.610668    3680 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_470', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793229667.955099    3681 dot_merger.cc:481] Merg

703/703 ━━━━━━━━━━━━━━━━━━━━ 57s 51ms/step - acc: 0.8327 - loss: 0.3824 - val_acc: 0.8868 - val_loss: 0.2793
Epoch 2/2
703/703 ━━━━━━━━━━━━━━━━━━━━ 30s 43ms/step - acc: 0.9191 - loss: 0.2246 - val_acc: 0.8836 - val_loss: 0.3146


I0000 00:00:1793229702.575101    3683 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_446', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793229707.955517    3684 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_440', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793229711.575165    3685 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_162575__.63
I0000 00:00:1793229712.771995    3685 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_476', 32 bytes spill stores, 32 bytes spill loads




Best epoch: 1
Validation loss (restored model): 0.2794
Validation accuracy: 0.8868
Validation F1: 0.8872

Validation confusion matrix:
[[1104  146]
 [ 137 1113]]

Training time: 00:01:28


### Graded Questions

In [16]:
# Set a1a to the validation accuracy at the epoch with minimum
# validation loss for the configuration with the lowest validation loss.

a1a = 0.8880                                       # Replace 0.0 with your answer

In [17]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a1a = {a1a:.4f}')

a1a = 0.8880


### Question a1b:

* How did validation accuracy change across the three context lengths (128 → 256 → 512)? Were the improvements equally large?
* How did training time change as context length increased?
* Compare the best Problem 1 configuration with the initial DistilBERT baseline. How much did validation accuracy improve, and what additional training time was required?
* Which configuration would you choose, considering both validation performance and computational cost? Explain your reasoning.

**Note:** Because learning rate and batch size also changed, you cannot attribute the differences in performance solely to context length.


Baseline

Best epoch: 1
Validation loss (restored model): 0.3561
Validation accuracy: 0.8456
Validation F1: 0.8457

Validation confusion matrix:
[[1056  194]
 [ 192 1058]]

1 min 19 seconds

---- 
Max Len 128, LR 2e-5, Batch Size 32:

Best epoch: 1
Validation loss (restored model): 0.3373
Validation accuracy: 0.8516
Validation F1: 0.8502

Validation confusion matrix:
[[1076  174]
 [ 197 1053]]

Training time: 00:01:03



-----
Max Len 256, LR 1e-5, Batch Size 16

Best epoch: 2
Validation loss (restored model): 0.2939
Validation accuracy: 0.8880
Validation F1: 0.8879

Validation confusion matrix:
[[1111  139]
 [ 141 1109]]

Training time: 00:01:51
------
Max Len 512, LR 7.5e-6, Batch Size 8

Best epoch: 1
Validation loss (restored model): 0.2794
Validation accuracy: 0.8868
Validation F1: 0.8872

Validation confusion matrix:
[[1104  146]
 [ 137 1113]]
 Training time: 00:01:28

In [8]:
a1b = """
The validation accuracies changed across all three context lengths. In experiment 1 (128), we received 0.8516 validation accuracy with 0.8502 F1 @ 0.3373 loss. 
In experiment 2 (256), these metrics boost to 0.8880, 0.8879, and 0.2939 respectively.
In experiment 3 (512), there is not much shift, with 0.8868 validation accuracy, 0.8872 validation F1, and 0.2794 loss.
The 2nd experiment performed best in terms of validation accuracy. The boost from 128 to 256 context length showed the largest performance boost.

The 128 context finished in 1 min and 3 seconds, the 256 context finished in 1 min and 51 seconds, and the 512 context finished in 1 min and 28 seconds. 
This shows that although context size increases with runtime, the changes to the learning rate and batch size also affected the training time.

Comparing the baseline with the best Problem 1 configuration, the baseline model had 0.8456 validation accuracy, 
whereas the 256 context length, our best performing model, had 0.8880 validation accuracy. 
This is a boost of 0.0424 at the expense of 30 extra seconds of training time. I would opt to chose the 256 context length configuration. Although there is a tradeoff
in training time, we can use greater computational resources to resolve this and get increased performance as a result.
"""



In [9]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a1b = {a1b}')

a1b = 
The validation accuracies changed across all three context lengths. In experiment 1 (128), we received 0.8516 validation accuracy with 0.8502 F1 @ 0.3373 loss. 
In experiment 2 (256), these metrics boost to 0.8880, 0.8879, and 0.2939 respectively.
In experiment 3 (512), there is not much shift, with 0.8868 validation accuracy, 0.8872 validation F1, and 0.2794 loss.
The 2nd experiment performed best in terms of validation accuracy. The boost from 128 to 256 context length showed the largest performance boost.

The 128 context finished in 1 min and 3 seconds, the 256 context finished in 1 min and 51 seconds, and the 512 context finished in 1 min and 28 seconds. 
This shows that although context size increases with runtime, the changes to the learning rate and batch size also affected the training time.

Comparing the baseline with the best Problem 1 configuration, the baseline model had 0.8456 validation accuracy, 
whereas the 256 context length, our best performing model, had 0.8

# Problem 2 — How Much Data Is Enough? (3 Runs)

In this problem, you'll investigate how the amount of training data affects the performance and computational cost of fine-tuning DistilBERT.

More training data generally improves predictive performance, but also increases training time. We want to determine whether the additional improvement is worth the computational cost.

## Setup

* Use a fixed DistilBERT configuration: `MAX_LEN = 256`, `LR = 1e-5`, and `BATCH = 16`, independently of the selected configuration in Problem 1.
* Keep the validation dataset fixed across all runs.
* Use `EPOCHS = 2` and EarlyStopping with `restore_best_weights=True`.
* Use the same random seed for all runs.

## Run these 3 configurations

| SUBSET_FRAC | Training data |
|---|---|
| 0.25 | 25% of the training set |
| 0.50 | 50% of the training set |
| 1.00 | 100% of the training set |

For each run, record:

* Best validation loss and the epoch at which it occurred.
* Validation accuracy at that epoch.
* Total training time.

Select the configuration with the lowest validation loss.

Then answer the graded questions.



In [12]:
p2_max_len = 256
p2_learning_rate = 1e-5
p2_batch_size = 16

### Run with 0.25 SUBSET_FRAC

In [13]:
print(f"Data mode: {DATA_MODE}")
print(f"Training examples: {len(X_tr):,}")
print(f"Validation examples: {len(X_va):,}")
print(f"Maximum sequence length: {p2_max_len}")
print(f"Batch size: {p2_batch_size}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)

preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length=p2_max_len
)

model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes=2,
    preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(p2_learning_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=p2_batch_size,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))


Data mode: baseline
Training examples: 5,624
Validation examples: 2,500
Maximum sequence length: 512
Batch size: 16
Epoch 1/2


/home/caleb/miniconda3/envs/tf_gpu/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1793233137.392591    3684 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_191456__.361
I0000 00:00:1793233146.538318    3684 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_2961', 32 bytes spill stores, 32 bytes spill loads



351/352 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - acc: 0.8323 - loss: 0.3844

I0000 00:00:1793233174.088537    3684 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_191456__.361
I0000 00:00:1793233183.269702    3684 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_2993', 32 bytes spill stores, 32 bytes spill loads



352/352 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - acc: 0.8321 - loss: 0.3847

I0000 00:00:1793233184.755912    3684 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_194874__.70
I0000 00:00:1793233186.084171    3684 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_440', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793233189.365200    3684 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_194874__.70
I0000 00:00:1793233191.482365    3684 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_482', 32 bytes spill stores, 32 bytes spill loads



352/352 ━━━━━━━━━━━━━━━━━━━━ 65s 128ms/step - acc: 0.8321 - loss: 0.3847 - val_acc: 0.8724 - val_loss: 0.3104
Epoch 2/2
352/352 ━━━━━━━━━━━━━━━━━━━━ 27s 77ms/step - acc: 0.9141 - loss: 0.2262 - val_acc: 0.8884 - val_loss: 0.2935


I0000 00:00:1793233221.576770    3681 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_446', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793233227.662645    3682 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_440', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793233231.372217    3683 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_203483__.63
I0000 00:00:1793233232.733555    3683 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_476', 32 bytes spill stores, 32 bytes spill loads




Best epoch: 2
Validation loss (restored model): 0.2935
Validation accuracy: 0.8884
Validation F1: 0.8884

Validation confusion matrix:
[[1111  139]
 [ 140 1110]]

Training time: 00:01:31


### Run with 0.50 SUBSET_FRAC

In [18]:
print(f"Data mode: {DATA_MODE}")
print(f"Training examples: {len(X_tr):,}")
print(f"Validation examples: {len(X_va):,}")
print(f"Maximum sequence length: {p2_max_len}")
print(f"Batch size: {p2_batch_size}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)

preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length=p2_max_len
)

model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes=2,
    preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(p2_learning_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=p2_batch_size,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))


Data mode: baselineplus
Training examples: 11,250
Validation examples: 2,500
Maximum sequence length: 256
Batch size: 16
Epoch 1/2


/home/caleb/miniconda3/envs/tf_gpu/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1793233429.910713    3680 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_232364__.361
I0000 00:00:1793233439.456128    3680 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_2961', 32 bytes spill stores, 32 bytes spill loads



703/704 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - acc: 0.8548 - loss: 0.3383

I0000 00:00:1793233490.850563    3682 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_232364__.361
I0000 00:00:1793233491.726882   28221 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_26', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1793233492.345273   28217 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_1_204', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1793233492.653727   28229 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_280', 60 bytes spill stores, 60 bytes spill loads

I0000 00:00:1793233494.024977   28217 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_67', 4 bytes spill stores, 8 bytes spill loads

I0000 00:00:1793233494.397084   28230 subprocess_compilation

704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - acc: 0.8548 - loss: 0.3383

I0000 00:00:1793233507.184004    3682 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_237190__.70
I0000 00:00:1793233508.388007    3682 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_440', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793233511.447657    3681 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_237190__.70
I0000 00:00:1793233512.795899    3681 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_482', 32 bytes spill stores, 32 bytes spill loads



704/704 ━━━━━━━━━━━━━━━━━━━━ 95s 104ms/step - acc: 0.8548 - loss: 0.3383 - val_acc: 0.8924 - val_loss: 0.2599
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 52s 74ms/step - acc: 0.9138 - loss: 0.2191 - val_acc: 0.8960 - val_loss: 0.2734


I0000 00:00:1793233567.882144    3681 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_446', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793233573.426752    3685 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_440', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793233576.898930    3685 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_246999__.63
I0000 00:00:1793233578.058983    3685 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_476', 32 bytes spill stores, 32 bytes spill loads




Best epoch: 1
Validation loss (restored model): 0.2599
Validation accuracy: 0.8924
Validation F1: 0.8925

Validation confusion matrix:
[[1114  136]
 [ 133 1117]]

Training time: 00:02:23


### Run with 1.00 SUBSET_FRAC

In [21]:
print(f"Data mode: {DATA_MODE}")
print(f"Training examples: {len(X_tr):,}")
print(f"Validation examples: {len(X_va):,}")
print(f"Maximum sequence length: {p1c_max_len}")
print(f"Batch size: {p2_batch_size}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)

preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length=p2_max_len
)

model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes=2,
    preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(p2_learning_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=p2_batch_size,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))


Data mode: full
Training examples: 22,500
Validation examples: 2,500
Maximum sequence length: 512
Batch size: 16
Epoch 1/2


/home/caleb/miniconda3/envs/tf_gpu/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1793233667.365993    3683 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_275880__.361
I0000 00:00:1793233676.883510    3683 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_2961', 32 bytes spill stores, 32 bytes spill loads



1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - acc: 0.8744 - loss: 0.3001

I0000 00:00:1793233778.288496    3683 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_275880__.361
I0000 00:00:1793233781.014036   30042 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_26', 12 bytes spill stores, 12 bytes spill loads

I0000 00:00:1793233781.297022   30051 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_280', 64 bytes spill stores, 64 bytes spill loads

I0000 00:00:1793233781.650846   30043 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_27', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1793233782.612482   30052 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_282', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1793233783.020922   30045 subprocess_compila

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - acc: 0.8744 - loss: 0.3000

I0000 00:00:1793233796.227336    3683 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_283518__.70
I0000 00:00:1793233797.578770    3683 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_440', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793233800.674233    3684 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_283518__.70
I0000 00:00:1793233801.925190    3684 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_482', 32 bytes spill stores, 32 bytes spill loads



1407/1407 ━━━━━━━━━━━━━━━━━━━━ 147s 89ms/step - acc: 0.8744 - loss: 0.3000 - val_acc: 0.8964 - val_loss: 0.2467
Epoch 2/2
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 100s 71ms/step - acc: 0.9236 - loss: 0.1990 - val_acc: 0.9036 - val_loss: 0.2402


I0000 00:00:1793233904.919269    3681 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_446', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793233910.359341    3681 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_440', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793233913.968840    3680 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_296347__.63
I0000 00:00:1793233915.197780    3680 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_476', 32 bytes spill stores, 32 bytes spill loads




Best epoch: 2
Validation loss (restored model): 0.2402
Validation accuracy: 0.9036
Validation F1: 0.9060

Validation confusion matrix:
[[1097  153]
 [  88 1162]]

Training time: 00:04:00


### Graded Questions

0.25 data size:

Best epoch: 2
Validation loss (restored model): 0.2935
Validation accuracy: 0.8884
Validation F1: 0.8884

Validation confusion matrix:
[[1111  139]
 [ 140 1110]]

Training time: 00:01:31

-----

0.50 data size:

Best epoch: 1
Validation loss (restored model): 0.2599
Validation accuracy: 0.8924
Validation F1: 0.8925

Validation confusion matrix:
[[1114  136]
 [ 133 1117]]

Training time: 00:02:23

-----


1.00 data size:

Best epoch: 2
Validation loss (restored model): 0.2402
Validation accuracy: 0.9036
Validation F1: 0.9060

Validation confusion matrix:
[[1097  153]
 [  88 1162]]

Training time: 00:04:00

In [22]:
# Set a2a to the validation accuracy at the epoch with minimum
# validation loss for the configuration with the lowest validation loss.

a2a = 0.9036  # Replace 0.0 with your answer

In [23]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a2a = {a2a:.4f}')

a2a = 0.9036


### Question a2b:

* How did validation accuracy change as the training dataset increased from 25% to 50% to 100%? Were the improvements equally large?
* How did training time change as the dataset increased?
* Which dataset size would you choose, considering both predictive performance and computational cost? Explain your reasoning using your results.


In [25]:
a2b = """
The validation accuracy changed unequally across each dataset size increase. At 0.25 data size validation accuracy was 0.8884. At 0.50 it reaches 0.8924. And at 1.00 it reaches 0.9036. 
The increases were ~0.004 and ~0.011. Therefore, the improvements were not equally large. The jump from 50% to 100% yielded the largest increase.

Training time increased across each dataset size. 25% took 1 minute 30 seconds, 50% took 2 minutes 23 seconds, and 100% took 4 minutes. The training time increased pretty linearly with the dataset size.

Given the performance at each dataset size, I would chose the full 1.00 dataset size. Given that the performance increased significantly about ~0.015 valdation accuracy at the cost of only a few additonal minutes
runtime on a 4060ti GPU, the time is not sigificant enough to warrant using a smaller dataset.
"""



In [26]:
## Graded Answer
# DO NOT change this cell in any way

print(f'a2b = {a2b}')

a2b = 
The validation accuracy changed unequally across each dataset size increase. At 0.25 data size validation accuracy was 0.8884. At 0.50 it reaches 0.8924. And at 1.00 it reaches 0.9036. 
The increases were ~0.004 and ~0.011. Therefore, the improvements were not equally large. The jump from 50% to 100% yielded the largest increase.

Training time increased across each dataset size. 25% took 1 minute 30 seconds, 50% took 2 minutes 23 seconds, and 100% took 4 minutes. The training time increased pretty linearly with the dataset size.

Given the performance at each dataset size, I would chose the full 1.00 dataset size. Given that the performance increased significantly about ~0.015 valdation accuracy at the cost of only a few additonal minutes
runtime on a 4060ti GPU, the time is not sigificant enough to warrant using a smaller dataset.



# Problem 3 — Model Swap: Speed vs. Accuracy (2 Runs)

In this problem, we'll compare two pretrained Transformer models with different capacities:

* **DistilBERT** — a smaller, computationally efficient model.
* **BERT-base** — a larger model with greater capacity, but higher computational cost.

Both models can be fine-tuned for the same text classification task. Our goal is to determine whether the larger model's improvement in predictive performance justifies its additional training time.

## Setup

Use the following settings for both models:

* `MAX_LEN = 256`
* `LR = 1e-5`
* `BATCH = 16`
* `SUBSET_FRAC = 0.50`
* `EPOCHS = 2`
* EarlyStopping with `restore_best_weights=True`
* Same random seed
* Same training and validation examples

Only the model architecture changes.

## Run these 2 configurations

| Model | Pretrained preset |
|---|---|
| DistilBERT | `distil_bert_base_en_uncased` |
| BERT-base | `bert_base_en_uncased` |

## How to switch models

**DistilBERT:**

```python
preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased", sequence_length=MAX_LEN
)
model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased", num_classes=2, preprocessor=preproc
)
```

**BERT-base:**

```python
preproc = kh.models.BertTextClassifierPreprocessor.from_preset(
    "bert_base_en_uncased", sequence_length=MAX_LEN
)
model = kh.models.BertTextClassifier.from_preset(
    "bert_base_en_uncased", num_classes=2, preprocessor=preproc
)
```

## What to do

Train and evaluate both models using the same experimental settings.

For each model, record:

* Best validation loss and the epoch at which it occurred.
* Validation accuracy at that epoch.
* Total training time.

Compare the results, then answer the graded questions.

**Note:** This experiment compares models under the same training budget, not necessarily at their individually optimal hyperparameters.


In [29]:
p3_max_len = 256
p3_lr = 1e-5
p3_batch = 16

### Distl Bert

In [ ]:
print(f"Data mode: {DATA_MODE}")
print(f"Training examples: {len(X_tr):,}")
print(f"Validation examples: {len(X_va):,}")
print(f"Maximum sequence length: {p3_max_len}")
print(f"Batch size: {p3_batch}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)

preproc = kh.models.DistilBertTextClassifierPreprocessor.from_preset(
    "distil_bert_base_en_uncased",
    sequence_length=p3_max_len
)

model = kh.models.DistilBertTextClassifier.from_preset(
    "distil_bert_base_en_uncased",
    num_classes=2,
    preprocessor=preproc
)


model.compile(
    optimizer=keras.optimizers.Adam(p3_lr),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=p3_batch,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))


Data mode: baselineplus
Training examples: 11,250
Validation examples: 2,500
Maximum sequence length: 256
Batch size: 16
Epoch 1/2


/home/caleb/miniconda3/envs/tf_gpu/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1793234532.302469    3680 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_325228__.361
I0000 00:00:1793234542.738492    3680 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_2961', 32 bytes spill stores, 32 bytes spill loads



703/704 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - acc: 0.8551 - loss: 0.3384

I0000 00:00:1793234595.371189    3685 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_325228__.361
I0000 00:00:1793234605.463157    3685 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_2993', 32 bytes spill stores, 32 bytes spill loads



704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - acc: 0.8551 - loss: 0.3383

I0000 00:00:1793234607.143215    3685 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_330054__.70
I0000 00:00:1793234608.489067    3685 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_440', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793234611.745145    3681 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_330054__.70
I0000 00:00:1793234613.048454    3681 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_482', 32 bytes spill stores, 32 bytes spill loads



704/704 ━━━━━━━━━━━━━━━━━━━━ 95s 100ms/step - acc: 0.8551 - loss: 0.3383 - val_acc: 0.8936 - val_loss: 0.2599
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 53s 75ms/step - acc: 0.9135 - loss: 0.2192 - val_acc: 0.8956 - val_loss: 0.2733


I0000 00:00:1793234669.088739    3682 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_446', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793234674.680997    3685 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_440', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793234678.310887    3680 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_339863__.63
I0000 00:00:1793234679.591511    3680 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_476', 32 bytes spill stores, 32 bytes spill loads




Best epoch: 1
Validation loss (restored model): 0.2599
Validation accuracy: 0.8936
Validation F1: 0.8938

Validation confusion matrix:
[[1115  135]
 [ 131 1119]]

Training time: 00:02:23


### Bert Base

In [35]:
print(f"Data mode: {DATA_MODE}")
print(f"Training examples: {len(X_tr):,}")
print(f"Validation examples: {len(X_va):,}")
print(f"Maximum sequence length: {p3_max_len}")
print(f"Batch size: {p3_batch}")

# Build preprocessor and classifier
keras.utils.set_random_seed(SEED)


preproc = kh.models.BertTextClassifierPreprocessor.from_preset(
    "bert_base_en_uncased", sequence_length=p3_max_len
)
model = kh.models.BertTextClassifier.from_preset(
    "bert_base_en_uncased", num_classes=2, preprocessor=preproc
)

model.compile(
    optimizer=keras.optimizers.Adam(p3_lr),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")]
)

# ---- Train with early stopping ----

cb = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

start = time.perf_counter()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=p3_batch,
    callbacks=cb,
    verbose=1
)

elapsed = time.perf_counter() - start

# ---- Evaluate the restored model on validation data ----

val_loss, val_acc = model.evaluate(
    X_va, y_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

logits = model.predict(
    X_va,
    batch_size=EVAL_BATCH,
    verbose=0
)

y_pred = np.argmax(logits, axis=-1)

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

acc = acc_metric.compute(
    predictions=y_pred,
    references=y_va
)["accuracy"]

f1 = f1_metric.compute(
    predictions=y_pred,
    references=y_va
)["f1"]

# ---- Confusion matrix ----

def confusion_matrix_np(y_true, y_pred, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

# ---- Summary ----

best_epoch = np.argmin(history.history["val_loss"]) + 1

print(f"\nBest epoch: {best_epoch}")
print(f"Validation loss (restored model): {val_loss:.4f}")
print(f"Validation accuracy: {acc:.4f}")
print(f"Validation F1: {f1:.4f}")

print("\nValidation confusion matrix:")
print(confusion_matrix_np(y_va, y_pred))

print("\nTraining time:",
      time.strftime("%H:%M:%S", time.gmtime(elapsed)))


Data mode: baselineplus
Training examples: 11,250
Validation examples: 2,500
Maximum sequence length: 256
Batch size: 16
Epoch 1/2


/home/caleb/miniconda3/envs/tf_gpu/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1793235426.382173    3682 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_466783__.689
I0000 00:00:1793235441.888585    3682 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_5468', 32 bytes spill stores, 32 bytes spill loads



703/704 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - acc: 0.8666 - loss: 0.3133

I0000 00:00:1793235538.232041    3682 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_466783__.689
I0000 00:00:1793235553.034757    3682 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_5566', 32 bytes spill stores, 32 bytes spill loads



704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - acc: 0.8667 - loss: 0.3132

I0000 00:00:1793235556.881778    3680 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_473451__.130
I0000 00:00:1793235558.416772    3680 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_437', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793235564.390236    3682 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_473451__.130
I0000 00:00:1793235566.078853    3682 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_509', 32 bytes spill stores, 32 bytes spill loads



704/704 ━━━━━━━━━━━━━━━━━━━━ 164s 177ms/step - acc: 0.8667 - loss: 0.3132 - val_acc: 0.9056 - val_loss: 0.2394
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 95s 135ms/step - acc: 0.9310 - loss: 0.1834 - val_acc: 0.9028 - val_loss: 0.2759


I0000 00:00:1793235665.031167    3679 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_437', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793235674.126519    3681 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_431', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1793235681.125261    3683 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_486431__.123
I0000 00:00:1793235682.649752    3683 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_503', 32 bytes spill stores, 32 bytes spill loads




Best epoch: 1
Validation loss (restored model): 0.2394
Validation accuracy: 0.9056
Validation F1: 0.9081

Validation confusion matrix:
[[1098  152]
 [  84 1166]]

Training time: 00:04:11


### Graded Questions

Distl Bert:

Best epoch: 1
Validation loss (restored model): 0.2599
Validation accuracy: 0.8936
Validation F1: 0.8938

Validation confusion matrix:
[[1115  135]
 [ 131 1119]]

Training time: 00:02:23

-----

Bert Base:

Best epoch: 1
Validation loss (restored model): 0.2394
Validation accuracy: 0.9056
Validation F1: 0.9081

Validation confusion matrix:
[[1098  152]
 [  84 1166]]

Training time: 00:04:11

In [38]:
# Set a3a to the validation accuracy at the epoch with minimum
# validation loss for the model with the lowest validation loss.

a3a = 0.9056  # Replace 0.0 with your answer

In [39]:
# Graded Answer
# DO NOT change this cell in any way

print(f'a3a = {a3a:.4f}')

a3a = 0.9056


### Question a3b:

* Which model achieved the higher validation accuracy? How large was the difference?
* How did their total training times compare?
* Given limited computational resources, which model would you choose? Explain whether the improvement in predictive performance justifies the additional computational cost.


In [36]:
a3b = """

The Distl Bert model validation accuracy at the minimum loss was 0.8936, whereas the Bert Base model's was 0.9056. This was an unexpected result, as increasing model complexity
did not significantly boost the validation accuracy. The overall difference in accuracy was 0.0120.

The training time of the Distl Bert was 2 minutes 23 seconds, whereas the Bert Base was 4 minutes 15 seconds.

Under the current runtime execution on Nvidia 4060ti, training times in single digit minutes are small and either model can be chosen regardless of training time. In that 
case I would chose the Bert model since it achieved 0.0120 higher validation accuracy than the Distl Bert. Given limited computational resources though I would chose the 
Distl Bert model. For less than half the training time it achieves extremely close relative performance. On less powerful hardware, this training time doubling could
make it significantly more difficult to fine tune the model architecture further, since experiment iteration takes longer.

"""



In [37]:
## Graded Answer
# DO NOT change this cell in any way

print(f'a3b = {a3b}')

a3b = 

The Distl Bert model validation accuracy at the minimum loss was 0.8936, whereas the Bert Base model's was 0.9056. This was an unexpected result, as increasing model complexity
did not significantly boost the validation accuracy. The overall difference in accuracy was 0.0120.

The training time of the Distl Bert was 2 minutes 23 seconds, whereas the Bert Base was 4 minutes 15 seconds.

Under the current runtime execution on Nvidia 4060ti, training times in single digit minutes are small and either model can be chosen regardless of training time. In that 
case I would chose the Bert model since it achieved 0.0120 higher validation accuracy than the Distl Bert. Given limited computational resources though I would chose the 
Distl Bert model. For less than half the training time it achieves extremely close relative performance. On less powerful hardware, this training time doubling could
make it significantly more difficult to fine tune the model architecture further, since experi